# Preparació dels jobs d'AlphaFold3 - BBF-14

Aquest notebook genera els inputs i els scripts d'enviament al clúster MareNostrum 5 (MN5) per a les prediccions estructurals amb AlphaFold3 dels 14 complexos target–binder del dataset BBF-14.

El fitxer d'entrada és un FASTA on cada entrada conté la seqüència del target (BBF-14, 122 aminoàcids, extreta del PDB amb codi 9HAG) i la del binder corresponent separades per dos punts. La classe sequenceModels de la llibreria prepare_proteins carrega i valida les seqüències directament des del FASTA; el separador : entre cadenes genera un avís de caràcter no estàndard que es pot ignorar, ja que és el format esperat per AlphaFold3.

La funció setUpAlphaFold3 crea l'estructura de carpetes del projecte (af3_jobs_bbf14), genera els fitxers d'entrada en format JSON per a cada complex i escriu les comandes d'execució corresponents. Les comandes no s'executen directament des d'aquí, sinó que es passen a mn5.jobArrays, que genera el fitxer de submissió SLURM (submit_af3_bbf14.sh) configurat per a la partició acc_bscls del MN5. El job s'envia al clúster amb sbatch submit_af3_bbf14.sh.

In [5]:
import sys
from pathlib import Path

base = Path.cwd().parent
sys.path.append(str(base))

from prepare_proteins import sequenceModels
from bsc_calculations.bsc_calculations import mn5

fasta_file = base / "inputs" / "input_af3_bbf14.fasta"
af3_folder = base / "scripts" / "af3_jobs_bbf14"
slurm_script = base / "scripts" / "submit_af3_bbf14.sh"

skip_finished = True
model_seeds = [1]

sequence_models = sequenceModels(str(fasta_file))

af3_jobs = sequence_models.setUpAlphaFold3(
    str(af3_folder),
    model_seeds=model_seeds,
    skip_finished=skip_finished,
)

if af3_jobs:
    mn5.jobArrays(
        af3_jobs,
        script_name=str(slurm_script),
        job_name='af3_bbf14',
        program='alphafold3',
    )

    print(f'Prepared {len(af3_jobs)} AF3 jobs in {af3_folder.resolve()}')
    print(f'Wrote {slurm_script}')
    print(f'Submit with: sbatch {slurm_script}')
    print(af3_jobs[0])

Prepared 14 AF3 jobs in /Users/bertaguiu/projects/scripts/af3_jobs_bbf14
Wrote /Users/bertaguiu/projects/scripts/submit_af3_bbf14.sh
Submit with: sbatch /Users/bertaguiu/projects/scripts/submit_af3_bbf14.sh
cd /Users/bertaguiu/projects/scripts/af3_jobs_bbf14/Complex1
bsc_alphafold input output $WEIGHTS
sbatch runner input
cd ../..



/opt/anaconda3/envs/prepare_proteins/lib/python3.10/site-packages/prepare_proteins-0.0.1-py3.10.egg/prepare_proteins/_prepare_sequences.py:160: UserWarning: Sequences contain non-standard amino acid codes — Complex1: :, Complex2: :, Complex3: :, Complex4: :, Complex5: :, Complex6: :, Complex7: :, Complex8: :, Complex9: :, Complex10: :, Complex11: :, Complex12: :, Complex13: :, Complex14: :
  warnings.warn(
